# Introduction to BERT
## 1. Loading Pre-trained BERT
In this section, we load the `bert-base-uncased` tokenizer and model from Hugging Face. We will tokenize a sample sentence to understand how BERT processes text into input IDs and attention masks, and then we will pass it through the model to get the contextualized embeddings.

In [4]:
from transformers import BertTokenizer, BertModel
import torch

# 1. Load Pre-trained Tokenizer and Model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 2. Tokenize Input Text
text = "Artificial Intelligence is transforming the world of technology."
inputs = tokenizer(text, return_tensors="pt")

print("Original Text:", text)
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
print("Input IDs:", inputs['input_ids'])
print("Attention Mask:", inputs['attention_mask'])

# 3. Generate Contextual Output
outputs = model(**inputs)
last_hidden_states = outputs.last_hidden_state

print("\nHidden state shape:", last_hidden_states.shape)
print("Meaning of shape: [Batch Size, Sequence Length, Hidden Size (768 for BERT-base)]")

Original Text: Artificial Intelligence is transforming the world of technology.
Tokens: ['[CLS]', 'artificial', 'intelligence', 'is', 'transforming', 'the', 'world', 'of', 'technology', '.', '[SEP]']
Input IDs: tensor([[  101,  7976,  4454,  2003, 17903,  1996,  2088,  1997,  2974,  1012,
           102]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

Hidden state shape: torch.Size([1, 11, 768])
Meaning of shape: [Batch Size, Sequence Length, Hidden Size (768 for BERT-base)]


## 2. Custom PyTorch Architecture for Classification
Here, we wrap the pre-trained BERT model in a custom PyTorch `nn.Module`. 
We freeze the base BERT layers so their weights don't update during training (transfer learning). Then, we add a dropout layer for regularization, a linear layer to downscale the embeddings from 768 to 512, and a final classification layer.

In [5]:
from torch import nn
from transformers import AutoModel

# Load model and freeze pre-trained layers
bert = AutoModel.from_pretrained('bert-base-uncased')
for param in bert.parameters():
    param.requires_grad = False

class BERT_architecture(nn.Module):
    def __init__(self, bert):
        super(BERT_architecture, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, 2) # 2 output classes
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, sent_id, mask):
        _, cls_hs = self.bert(sent_id, attention_mask=mask, return_dict=False)
        x = self.dropout(torch.relu(self.fc1(cls_hs)))
        return self.softmax(self.fc2(x))

# Instantiate the custom model
custom_model = BERT_architecture(bert)
print("Custom Model Architecture Initialized!")

# Test it with the inputs from the previous cell
print("\nTesting the custom model with earlier tokenized input:")
dummy_output = custom_model(inputs['input_ids'], inputs['attention_mask'])
print("Log Softmax Output Shape (Classes = 2):", dummy_output.shape)
print("Output Values (Log Probabilities):", dummy_output)

Custom Model Architecture Initialized!

Testing the custom model with earlier tokenized input:
Log Softmax Output Shape (Classes = 2): torch.Size([1, 2])
Output Values (Log Probabilities): tensor([[-0.8760, -0.5386]], grad_fn=<LogSoftmaxBackward0>)


## 3. Named Entity Recognition (NER)
Named Entity Recognition extracts structured information from unstructured text, classifying entities into categories like Organizations, Locations, and Persons. 
We will use Hugging Face's pipeline for out-of-the-box entity detection and see how it performs on a sample sentence.

In [6]:
from transformers import pipeline

# Initialize NER pipeline with a pre-trained NER model
print("Loading NER pipeline...")
ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

text = "Microsoft Corporation is headquartered in Redmond, Washington, USA."
print("\nInput Text:", text)
print("-" * 50)

entities = ner(text)

# Extract and print identified entities
print("Identified Entities:")
for entity in entities:
    print(f"[{entity['entity_group']}] -> '{entity['word']}' (Confidence: {entity['score']:.4f})")

Loading NER pipeline...


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu



Input Text: Microsoft Corporation is headquartered in Redmond, Washington, USA.
--------------------------------------------------
Identified Entities:
[ORG] -> 'Microsoft Corporation' (Confidence: 0.9992)
[LOC] -> 'Redmond' (Confidence: 0.9664)
[LOC] -> 'Washington' (Confidence: 0.9991)
[LOC] -> 'USA' (Confidence: 0.9991)
